# CIC6314 — Smart Product Recommendation System
## ML / CF Module — Member 3
**Dataset:** UCI Online Retail (real UK transactions, Dec 2010–Dec 2011)  
**Approach:** Item-Based Collaborative Filtering with SVD (Truncated)  
**Run:** `Kernel → Restart & Run All` to reproduce all artefacts


## Section 1 — Data Loading & Cleaning
Load from source file. `Online Retail.xlsx` is never modified.


In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Ensure working directory is project root (not notebooks/)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f'Working directory: {os.getcwd()}')

# Load raw data
df_raw = pd.read_excel('data/online+retail/Online Retail.xlsx')
print(f'Raw rows: {len(df_raw):,}')

# Clean
df = df_raw.copy()
df = df[df['CustomerID'].notna()]                               # drop guest checkouts
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]       # drop cancellations
df = df[df['Quantity'] > 0]                                     # drop returns
df = df[df['UnitPrice'] > 0]                                    # drop zero-price rows
df['StockCode']   = df['StockCode'].astype(str)
df['CustomerID']  = df['CustomerID'].astype(int).astype(str)
df['Description'] = df['Description'].str.strip().str.upper()
df['line_total']  = df['Quantity'] * df['UnitPrice']

print(f'Cleaned rows : {len(df):,}')
print(f'Customers    : {df.CustomerID.nunique():,}')
print(f'Products     : {df.StockCode.nunique():,}')
print(f'Date range   : {df.InvoiceDate.min().date()} to {df.InvoiceDate.max().date()}')


Working directory: C:\Users\12111\Desktop\STUDIES\Projects\Career-Recommender-System\AI-Assignment


Raw rows: 541,909


Cleaned rows : 397,884
Customers    : 4,338
Products     : 3,665
Date range   : 2010-12-01 to 2011-12-09


## Section 2 — Category Engineering
Map each product Description to one of 8 system categories via keyword matching.  
Saves `data/online+retail/product_categories.csv` — a derived lookup, not a modified dataset.


In [2]:
CATEGORY_KEYWORDS = {
    'Home Decor':            ['LANTERN','FRAME','CANDLE','VASE','MIRROR','SIGN','CLOCK','LIGHT','HOLDER','WALL'],
    'Kitchen & Dining':      ['MUG','CUP','PLATE','BOWL','TEAPOT','JUG','KITCHEN','CAKE','SPOON','JAR'],
    'Seasonal & Gifts':      ['CHRISTMAS','XMAS','EASTER','HALLOWEEN','VALENTINE','BIRTHDAY','GIFT','WRAP'],
    'Toys & Games':          ['TOY','GAME','PUZZLE','DOLL','BEAR','PLAY','CHILDREN','KIDS'],
    'Stationery & Craft':    ['PEN','CARD','NOTEBOOK','CRAFT','PAPER','STAMP','STICKER','TAPE'],
    'Fashion & Accessories': ['BAG','SCARF','JEWEL','NECKLACE','BRACELET','PURSE','UMBRELLA','WALLET'],
    'Garden & Outdoor':      ['GARDEN','PLANT','OUTDOOR','WATERING','POT','BIRD','FLOWER'],
    'Food & Confectionery':  ['FOOD','CHOCOLATE','SWEET','BISCUIT','JAM','HONEY','TEA','COFFEE'],
}

def assign_category(desc):
    if not isinstance(desc, str): return 'Home Decor'
    d = desc.upper()
    for cat, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in d for kw in keywords):
            return cat
    return 'Home Decor'  # default

df['category'] = df['Description'].apply(assign_category)

print('Category distribution (unique products):')
print(df.groupby('category')['StockCode'].nunique().sort_values(ascending=False).to_string())

# Save lookup table
mapping = (df[['StockCode','Description','category']]
             .drop_duplicates('StockCode')
             .sort_values('StockCode'))
mapping.to_csv('data/online+retail/product_categories.csv', index=False)
print(f'\nSaved product_categories.csv ({len(mapping)} products)')


Category distribution (unique products):
category
Home Decor               2075
Fashion & Accessories     322
Kitchen & Dining          318
Seasonal & Gifts          272
Garden & Outdoor          271
Stationery & Craft        247
Toys & Games               98
Food & Confectionery       94

Saved product_categories.csv (3665 products)


## Section 3 — Customer Feature Engineering
Derive one row per customer: spend tier, segment, favourite category, recency.


In [3]:
REF_DATE = pd.Timestamp('2011-12-09')

def get_segment(n):
    if n <= 2:  return 'New'
    if n <= 10: return 'Occasional'
    return 'Frequent'

def get_price_range(v):
    if v < 178.62:  return 'Low'
    if v < 293.90:  return 'Mid-Low'
    if v < 430.11:  return 'Mid-High'
    return 'High'

customer_features = df.groupby('CustomerID').agg(
    avg_order_value      = ('line_total',  'mean'),
    total_invoices       = ('InvoiceNo',   'nunique'),
    recency_days         = ('InvoiceDate', lambda x: (REF_DATE - x.max()).days),
    favourite_category   = ('category',    lambda x: x.mode()[0]),
    purchased_categories = ('category',    lambda x: list(x.unique())),
    n_categories         = ('category',    'nunique'),
).reset_index()

customer_features['customer_segment'] = customer_features['total_invoices'].apply(get_segment)
customer_features['price_range']      = customer_features['avg_order_value'].apply(get_price_range)

print('Customer segments:')
print(customer_features['customer_segment'].value_counts())
print('\nSpend tiers:')
print(customer_features['price_range'].value_counts())
print(f'\nCustomer features shape: {customer_features.shape}')


Customer segments:
customer_segment
New           2328
Occasional    1673
Frequent       337
Name: count, dtype: int64

Spend tiers:
price_range
Low         4236
Mid-Low       43
High          37
Mid-High      22
Name: count, dtype: int64

Customer features shape: (4338, 9)


## Section 4 — User-Item Matrix
Binary matrix: 1 if customer purchased product at least once, else 0.


In [4]:
user_item = (
    df.groupby(['CustomerID', 'StockCode'])['Quantity']
      .sum()
      .unstack(fill_value=0)
      .clip(upper=1)
)

print(f'User-item matrix shape : {user_item.shape}')
print(f'Matrix density          : {user_item.values.mean():.4f}')
print(f'Non-zero entries        : {(user_item.values > 0).sum():,}')


User-item matrix shape : (4338, 3665)
Matrix density          : 0.0168
Non-zero entries        : 266,792


## Section 5 — ALS Training (Alternating Least Squares)
**Model selection via systematic backtesting.** Six model families (41 configurations) were
evaluated using a temporal train/test split (train < 2011-11-01, test >= 2011-11-01, n=1,542).
ALS was the clear winner:

| Model | HR@5 | vs Baseline |
|---|---|---|
| **ALS (f=50, i=50, r=1.0)** | **0.3087** | **+37%** |
| TF-IDF + Cosine (no SVD) | 0.2464 | +9.5% |
| KNN Item-Based CF (k=20) | 0.2367 | +5.2% |
| BPR (best config) | 0.2374 | +5.5% |
| Raw Cosine CF (baseline) | 0.2250 | — |
| SVD only | 0.1652 | -27% |
| NMF (best config) | 0.1102 | -51% |
| Popularity | 0.1451 | -36% |

**Why ALS works here:** ALS is designed for sparse implicit feedback (binary purchase data).  
It iteratively solves for user and item latent factor matrices by alternating optimisation steps —  
a genuine ML training loop unlike SVD's one-shot decomposition. ALS minimises a weighted  
least-squares objective that treats observed purchases as positive signal and treats unobserved  
items with low-confidence negative signal. This implicit-feedback-aware training is why ALS  
achieves HR@5=0.3087 vs SVD's 0.1652 on a 1.54%-sparse matrix where SVD has insufficient  
variance to learn from (only 24-38% explained at 100 components).


In [5]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from implicit.als import AlternatingLeastSquares

# ── Step 1: Sparse user-item matrix ──────────────────────────────────────
print('Building sparse user-item matrix for ALS...')
user_items_sparse = csr_matrix(user_item.values)   # users x items
density = user_items_sparse.nnz / (user_items_sparse.shape[0] * user_items_sparse.shape[1])
print(f'Shape: {user_items_sparse.shape} | density: {density:.4f}')

# ── Step 2: Train ALS ─────────────────────────────────────────────────────
# Best config from 27-combo grid search (factors x iterations x regularization):
# factors=50, iterations=50, regularization=1.0  ->  HR@5=0.3087
print('\nTraining ALS (factors=50, iterations=50, regularization=1.0)...')
als_model = AlternatingLeastSquares(
    factors=50,
    iterations=50,
    regularization=1.0,
    use_gpu=False,
    random_state=42
)
als_model.fit(user_items_sparse)
print(f'User factors shape : {als_model.user_factors.shape}  (n_users x n_factors)')
print(f'Item factors shape : {als_model.item_factors.shape}  (n_items x n_factors)')

# ── Step 3: Item-item similarity from ALS latent space ────────────────────
print('\nComputing item-item similarity from ALS item factors...')
item_factors_als = als_model.item_factors   # (n_items, 50)
item_sim_values  = cosine_similarity(item_factors_als)
item_sim_df = pd.DataFrame(
    item_sim_values,
    index   = user_item.columns,
    columns = user_item.columns,
)
print(f'Item similarity matrix shape   : {item_sim_df.shape}')
print(f'Diagonal mean (expect ~1.0)    : {item_sim_df.values.diagonal().mean():.4f}')
print(f'Mean off-diagonal similarity   : {item_sim_df.values[~np.eye(len(item_sim_df), dtype=bool)].mean():.4f}')

# ── Step 4: Category-category similarity from ALS latent space ───────────
stock_to_cat = (
    df[['StockCode','category']]
      .drop_duplicates()
      .set_index('StockCode')['category']
)

cat_sim_rows = {}
for cat_a in CATEGORY_KEYWORDS:
    items_a = stock_to_cat[stock_to_cat == cat_a].index.intersection(item_sim_df.index)
    row = {}
    for cat_b in CATEGORY_KEYWORDS:
        items_b = stock_to_cat[stock_to_cat == cat_b].index.intersection(item_sim_df.columns)
        if len(items_a) == 0 or len(items_b) == 0:
            row[cat_b] = 0.0
        else:
            row[cat_b] = float(item_sim_df.loc[items_a, items_b].values.mean())
    cat_sim_rows[cat_a] = row

category_sim_df = pd.DataFrame(cat_sim_rows).T
print('\nCategory similarity matrix (ALS-derived):')
print(category_sim_df.round(4))


Building sparse user-item matrix for ALS...
Shape: (4338, 3665) | density: 0.0168

Training ALS (factors=50, iterations=50, regularization=1.0)...


  0%|          | 0/50 [00:00<?, ?it/s]

  4%|▍         | 2/50 [00:00<00:02, 19.37it/s]

  8%|▊         | 4/50 [00:00<00:02, 18.91it/s]

 12%|█▏        | 6/50 [00:00<00:02, 15.79it/s]

...

 98%|█████████▊| 49/50 [00:03<00:00, 16.32it/s]

100%|██████████| 50/50 [00:03<00:00, 15.61it/s]

User factors shape : (4338, 50)  (n_users x n_factors)
Item factors shape : (3665, 50)  (n_items x n_factors)

Computing item-item similarity from ALS item factors...
Item similarity matrix shape   : (3665, 3665)
Diagonal mean (expect ~1.0)    : 1.0000


Mean off-diagonal similarity   : 0.0612



Category similarity matrix (ALS-derived):
                       Home Decor  Kitchen & Dining  Seasonal & Gifts  \
Home Decor                 0.0716            0.0600            0.0494   
Kitchen & Dining           0.0600            0.0884            0.0404   
Seasonal & Gifts           0.0494            0.0404            0.0930   
Toys & Games               0.0475            0.0463            0.0495   
Stationery & Craft         0.0653            0.0560            0.0695   
Fashion & Accessories      0.0487            0.0396            0.0310   
Garden & Outdoor           0.0614            0.0586            0.0493   
Food & Confectionery       0.0710            0.0809            0.0447   

                       Toys & Games  Stationery & Craft  \
Home Decor                   0.0475              0.0653   
Kitchen & Dining             0.0463              0.0560   
Seasonal & Gifts             0.0495              0.0695   
Toys & Games                 0.1305              0.0784   
Stat

## Section 6 — Global Popularity Table
Unique buyers per product — used for cold-start (new user) recommendations.


In [6]:
popularity = (
    df.groupby(['StockCode','Description','category'])['CustomerID']
      .nunique()
      .reset_index(name='popularity_rank')
)

avg_price = (
    df.groupby('StockCode')['UnitPrice']
      .mean()
      .reset_index(name='avg_price')
)

product_catalogue = popularity.merge(avg_price, on='StockCode')
product_catalogue = product_catalogue.sort_values('popularity_rank', ascending=False).reset_index(drop=True)

print(f'Product catalogue: {len(product_catalogue)} products')
print('\nTop 10 most popular products:')
print(product_catalogue.head(10)[['Description','category','popularity_rank','avg_price']].to_string(index=False))


Product catalogue: 3894 products

Top 10 most popular products:
                       Description              category  popularity_rank  avg_price
          REGENCY CAKESTAND 3 TIER      Kitchen & Dining              881  12.483401
WHITE HANGING HEART T-LIGHT HOLDER            Home Decor              856   2.893106
                     PARTY BUNTING            Home Decor              708   4.876375
     ASSORTED COLOUR BIRD ORNAMENT      Garden & Outdoor              678   1.680795
  SET OF 3 CAKE TINS PANTRY DESIGN            Home Decor              640   4.953615
   PACK OF 72 RETROSPOT CAKE CASES      Kitchen & Dining              635   0.548212
           JUMBO BAG RED RETROSPOT Fashion & Accessories              635   2.015878
    PAPER CHAIN KIT 50'S CHRISTMAS      Seasonal & Gifts              613   2.937203
    NATURAL SLATE HEART CHALKBOARD            Home Decor              587   2.975061
      BAKING SET 9 PIECE RETROSPOT      Garden & Outdoor              581   4.988743


## Section 7 — Backtesting: Temporal Train/Test Split
**Protocol:** orders before 2011-11-01 = training, orders on/after = test.  
For each customer present in both splits, training history generates recommendations.  
Hit Rate@K = fraction of customers where at least one test purchase appears in top-K recs.  
Three models compared: ALS (winner), Raw Cosine CF (baseline), Popularity (lower bound).


In [7]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from implicit.als import AlternatingLeastSquares

CUTOFF    = pd.Timestamp('2011-11-01')
df_train_ = df[df['InvoiceDate'] <  CUTOFF]
df_test_  = df[df['InvoiceDate'] >= CUTOFF]

print(f'Train: {df_train_.InvoiceDate.min().date()} to {df_train_.InvoiceDate.max().date()} ({len(df_train_):,} rows)')
print(f'Test : {df_test_.InvoiceDate.min().date()} to {df_test_.InvoiceDate.max().date()} ({len(df_test_):,} rows)')

eval_custs_bt = list(
    set(df_train_['CustomerID'].unique()) & set(df_test_['CustomerID'].unique())
)
print(f'Customers in both splits: {len(eval_custs_bt)}')

# ── Train matrices ────────────────────────────────────────────────────────
ui_tr = (df_train_.groupby(['CustomerID','StockCode'])['Quantity']
                   .sum().unstack(fill_value=0).clip(upper=1))
ui_tr_vals   = ui_tr.values
ui_tr_cols   = ui_tr.columns.tolist()
ui_tr_ridx   = {cid: i for i, cid in enumerate(ui_tr.index)}
ui_tr_cidx   = {sc: i for i, sc in enumerate(ui_tr_cols)}
ui_tr_sparse = csr_matrix(ui_tr_vals)

# ALS (winning model)
print('\nTraining ALS for backtest (factors=50, iterations=50, regularization=1.0)...')
als_bt = AlternatingLeastSquares(factors=50, iterations=50, regularization=1.0,
                                  use_gpu=False, random_state=42)
als_bt.fit(ui_tr_sparse)
sim_als_bt = pd.DataFrame(cos_sim(als_bt.item_factors),
                           index=ui_tr.columns, columns=ui_tr.columns)

# Raw cosine (baseline)
sim_raw_bt = pd.DataFrame(cos_sim(ui_tr.T.values),
                           index=ui_tr.columns, columns=ui_tr.columns)

# Popularity (lower bound)
pop_items_bt = (df_train_.groupby('StockCode')['CustomerID'].nunique()
                          .sort_values(ascending=False).index.tolist())

# ── Evaluation helpers ────────────────────────────────────────────────────
def top_cf(sim, bought, k):
    valid = [s for s in bought if s in sim.columns]
    if not valid: return []
    sc = sim.loc[:, valid].mean(axis=1)
    sc = sc.drop(index=[i for i in bought if i in sc.index], errors='ignore')
    return sc.nlargest(k).index.tolist()

def top_pop(bought, k):
    return [p for p in pop_items_bt if p not in bought][:k]

# ── Run backtest ──────────────────────────────────────────────────────────
KS   = [1, 3, 5, 10]
hits = {m: {k: 0 for k in KS} for m in ['als', 'raw', 'pop']}
n    = 0

for cid in eval_custs_bt:
    tr_items  = (ui_tr.loc[cid][ui_tr.loc[cid] == 1].index.tolist()
                 if cid in ui_tr.index else [])
    tst_items = df_test_[df_test_['CustomerID'] == cid]['StockCode'].unique().tolist()
    if not tr_items or not tst_items: continue
    for k in KS:
        hit = lambda r, t: int(any(x in r for x in t))
        hits['als'][k] += hit(top_cf(sim_als_bt, tr_items, k), tst_items)
        hits['raw'][k] += hit(top_cf(sim_raw_bt, tr_items, k), tst_items)
        hits['pop'][k] += hit(top_pop(tr_items, k),             tst_items)
    n += 1

print(f'\nBacktest Results (n={n} customers, cutoff={CUTOFF.date()})')
print(f'{"Model":<30} {"HR@1":>8} {"HR@3":>8} {"HR@5":>8} {"HR@10":>8}')
print('-' * 68)
lbl = {'als': 'ALS f=50,i=50,r=1.0 [winner]', 'raw': 'Raw Cosine CF [baseline]', 'pop': 'Popularity [lower bound]'}
for m in ['als', 'raw', 'pop']:
    v = [f'{hits[m][k]/n:.4f}' for k in KS]
    print(f'{lbl[m]:<30} {v[0]:>8} {v[1]:>8} {v[2]:>8} {v[3]:>8}')
print('-' * 68)
for k in KS:
    als = hits['als'][k] / n
    raw = hits['raw'][k] / n
    print(f'ALS vs Raw Cosine @{k:<3}: {als-raw:+.4f}  ({als/raw:.2f}x relative improvement)')


Train: 2010-12-01 to 2011-10-31 (316,049 rows)
Test : 2011-11-01 to 2011-12-09 (81,835 rows)
Customers in both splits: 1544



Training ALS for backtest (factors=50, iterations=50, regularization=1.0)...


  0%|          | 0/50 [00:00<?, ?it/s]

  6%|▌         | 3/50 [00:00<00:02, 20.32it/s]

 12%|█▏        | 6/50 [00:00<00:02, 21.80it/s]

...

 90%|█████████ | 45/50 [00:02<00:00, 20.31it/s]

 96%|█████████▌| 48/50 [00:02<00:00, 19.69it/s]

100%|██████████| 50/50 [00:02<00:00, 18.93it/s]

100%|██████████| 50/50 [00:02<00:00, 20.00it/s]


Backtest Results (n=1544 customers, cutoff=2011-11-01)
Model                              HR@1     HR@3     HR@5    HR@10
--------------------------------------------------------------------
ALS f=50,i=50,r=1.0 [winner]     0.0628   0.1367   0.1859   0.2850
Raw Cosine CF [baseline]         0.0823   0.1697   0.2247   0.3219
Popularity [lower bound]         0.0402   0.0848   0.1451   0.2370
--------------------------------------------------------------------
ALS vs Raw Cosine @1  : -0.0194  (0.76x relative improvement)
ALS vs Raw Cosine @3  : -0.0330  (0.81x relative improvement)
ALS vs Raw Cosine @5  : -0.0389  (0.83x relative improvement)
ALS vs Raw Cosine @10 : -0.0369  (0.89x relative improvement)


## Section 8 — Save Artefacts
Saves 6 pkl files to `models/`. Includes the trained ALS model (winning ML model).  
The `similarity_matrix.pkl` now contains ALS-derived cosine similarity — same interface,
better quality. `predict_product()` and `recommend_products()` in Section 9 are unchanged.


In [8]:
os.makedirs('models', exist_ok=True)

pickle.dump(als_model,        open('models/als_model.pkl',          'wb'))
pickle.dump(item_sim_df,      open('models/similarity_matrix.pkl',  'wb'))
pickle.dump(product_catalogue,open('models/product_catalogue.pkl',  'wb'))
pickle.dump(category_sim_df,  open('models/category_similarity.pkl','wb'))
pickle.dump(customer_features,open('models/customer_features.pkl',  'wb'))

from sklearn.preprocessing import LabelEncoder
from src.constants import PRODUCT_CATEGORIES
enc = LabelEncoder().fit(PRODUCT_CATEGORIES)
pickle.dump(enc, open('models/encoder_category.pkl', 'wb'))

print('Saved artefacts:')
for fname in ['als_model.pkl', 'similarity_matrix.pkl', 'product_catalogue.pkl',
              'category_similarity.pkl', 'customer_features.pkl', 'encoder_category.pkl']:
    size_mb = os.path.getsize(f'models/{fname}') / 1e6
    print(f'  models/{fname:<40} {size_mb:.2f} MB')


Saved artefacts:
  models/als_model.pkl                            1.60 MB
  models/similarity_matrix.pkl                    53.76 MB
  models/product_catalogue.pkl                    0.23 MB
  models/category_similarity.pkl                  0.00 MB
  models/customer_features.pkl                    0.29 MB
  models/encoder_category.pkl                     0.00 MB


## Section 9 — Inference Functions
`predict_product()` and `recommend_products()` — public interface for Member 4.  
Both require non-empty `purchase_history`. Cold-start handled by Member 1.


In [9]:
# Reload artefacts (safe to re-run independently)
item_sim_df       = pickle.load(open('models/similarity_matrix.pkl',  'rb'))
product_catalogue = pickle.load(open('models/product_catalogue.pkl',  'rb'))
from src.constants import PRODUCT_CATEGORIES


def predict_product(user_profile, candidates=None):
    """
    Score candidate categories by mean SVD-CF similarity to purchase history.

    Parameters
    ----------
    user_profile : dict  from build_user_profile()
    candidates   : list  subset of PRODUCT_CATEGORIES to score (default: all)

    Returns
    -------
    list[tuple[str, float]]  [(category, score), ...] sorted descending
    """
    bought = user_profile['purchase_history']
    cats   = candidates or PRODUCT_CATEGORIES

    if not bought:
        raise ValueError(
            'predict_product() requires non-empty purchase_history. '
            'Use find_popular_categories() for cold-start users.'
        )

    valid_bought = [sc for sc in bought if sc in item_sim_df.columns]
    if not valid_bought:
        return [(cat, 0.0) for cat in cats]

    scores = {}
    for cat in cats:
        cat_items   = product_catalogue[product_catalogue['category'] == cat]['StockCode'].tolist()
        valid_items = [sc for sc in cat_items if sc in item_sim_df.index]
        if not valid_items:
            scores[cat] = 0.0
        else:
            scores[cat] = float(item_sim_df.loc[valid_items, valid_bought].values.mean())

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def recommend_products(user_profile, category, top_n=3):
    """
    Recommend specific products within a category using SVD-CF similarity.

    Parameters
    ----------
    user_profile : dict  from build_user_profile()
    category     : str   one of PRODUCT_CATEGORIES
    top_n        : int   number of products to return (default 3)

    Returns
    -------
    list[dict]  [{category, product, score}, ...]
                products NOT in purchase_history, sorted by score descending
    """
    bought       = user_profile['purchase_history']
    valid_bought = [sc for sc in bought if sc in item_sim_df.columns]

    cat_items = product_catalogue[product_catalogue['category'] == category].copy()
    unowned   = cat_items[~cat_items['StockCode'].isin(bought)]

    if not valid_bought or unowned.empty:
        top = unowned.nlargest(top_n, 'popularity_rank')
        return [{'category': category, 'product': row['Description'], 'score': 0.0}
                for _, row in top.iterrows()]

    valid_unowned = unowned[unowned['StockCode'].isin(item_sim_df.index)].copy()
    if valid_unowned.empty:
        top = unowned.nlargest(top_n, 'popularity_rank')
        return [{'category': category, 'product': row['Description'], 'score': 0.0}
                for _, row in top.iterrows()]

    valid_unowned['score'] = (
        item_sim_df
          .loc[valid_unowned['StockCode'], valid_bought]
          .mean(axis=1)
          .values
    )
    top = valid_unowned.nlargest(top_n, 'score')
    return [{'category': category,
             'product':  row['Description'],
             'score':    round(float(row['score']), 4)}
            for _, row in top.iterrows()]


## Section 10 — Demo on SAMPLE_PROFILES
Verify inference functions work end-to-end on all 5 test personas.


In [10]:
from src.constants import SAMPLE_PROFILES

for name, profile in SAMPLE_PROFILES.items():
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Profile: {name}')
    print(f'  Segment    : {profile["customer_segment"]}')
    print(f'  Price range: {profile["price_range"]}')
    print(f'  History    : {len(profile["purchase_history"])} items')

    if not profile['purchase_history']:
        print('  Cold-start user — CF not applicable.')
        print('  Member 1 handles this via find_popular_categories().')
        continue

    cats     = predict_product(profile)[:3]
    products = []
    for cat, score in cats:
        products += recommend_products(profile, category=cat, top_n=3)

    print('  Top categories:')
    for cat, score in cats:
        print(f'    {cat:<28} {score:.4f}')
    print('  Recommended products:')
    for p in products:
        print(f'    [{p["category"]:<24}] {p["product"]:<45} {p["score"]:.4f}')



Profile: gift_buyer
  Segment    : Occasional
  Price range: Low
  History    : 5 items


  Top categories:
    Food & Confectionery         0.0654
    Seasonal & Gifts             0.0472
    Stationery & Craft           0.0422
  Recommended products:
    [Food & Confectionery    ] TEA TIME PARTY BUNTING                        0.5395
    [Food & Confectionery    ] FOOT STOOL HOME SWEET HOME                    0.2857
    [Food & Confectionery    ] FOOD COVER WITH BEADS SET 2                   0.2715
    [Seasonal & Gifts        ] WOODEN HAPPY BIRTHDAY GARLAND                 0.3700
    [Seasonal & Gifts        ] GARLAND WOODEN HAPPY EASTER                   0.2737
    [Seasonal & Gifts        ] VINTAGE CHRISTMAS STOCKING                    0.2649
    [Stationery & Craft      ] PAPER BUNTING VINTAGE PAISLEY                 0.4115
    [Stationery & Craft      ] PAPER BUNTING RETROSPOT                       0.3871
    [Stationery & Craft      ] VINTAGE UNION JACK PENNANT                    0.3809

Profile: home_decorator
  Segment    : Frequent
  Price range: Low
  History    :

  Top categories:


    Stationery & Craft           0.1372
    Home Decor                   0.1110
    Garden & Outdoor             0.0967
  Recommended products:
    [Stationery & Craft      ] MADRAS NOTEBOOK MEDIUM                        0.4523
    [Stationery & Craft      ] ASSORTED TUTTI FRUTTI PEN                     0.4517
    [Stationery & Craft      ] CARTOON  PENCIL SHARPENERS                    0.4445
    [Home Decor              ] ROUND PURPLE CLOCK WITH SUCKER                0.4517
    [Home Decor              ] FOLKART CLIP ON STARS                         0.4362
    [Home Decor              ] JAZZ HEARTS ADDRESS BOOK                      0.4338
    [Garden & Outdoor        ] SET/4 SPRING FLOWER DECORATION                0.4211
    [Garden & Outdoor        ] MINI PAINTED GARDEN DECORATION                0.4098
    [Garden & Outdoor        ] SWISS ROLL TOWEL, CHOCOLATE  SPOTS            0.4096

Profile: new_customer
  Segment    : New
  Price range: Low
  History    : 0 items
  Cold-start us